In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")


@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [4]:
from dataclasses import dataclass

@dataclass
class UserRole:
    user_role: str = "external"

In [5]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role
    
    if user_role == "internal":
        pass # internal users get access to all tools
    else:
        tools = [web_search] # external users only get access to web search
        request = request.override(tools=tools) 

    return handler(request)

In [6]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],
    context_schema=UserRole
)

In [7]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "external"}
)

print(response["messages"][-1].content)

I don’t have enough context to know which database you’re referring to. Could you clarify:

- Which database or dataset are you using (e.g., PostgreSQL, MySQL, SQLite, MongoDB, a CSV file, etc.)?
- What is the table/collection name that stores artists (e.g., artists)?
- Do you want the exact count of rows, or the number of distinct artists (in case of duplicates or multiple records per artist)?
- Are there multiple tables that should be considered together (e.g., artists and artist_aliases)?

Here are quick queries you can use depending on your setup:

- SQL (PostgreSQL, MySQL, SQLite) - exact number of rows in artists:
  SELECT COUNT(*) AS artist_count FROM artists;

- SQL - distinct artists by id:
  SELECT COUNT(DISTINCT artist_id) AS artist_count FROM artists;

- SQL - distinct by name:
  SELECT COUNT(DISTINCT name) AS artist_count FROM artists;

- MongoDB - exact count (assuming a single collection of artists):
  db.artists.countDocuments({})

- If you have multiple tables/collecti